In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from gradient_descent import BCE, numerical_grad, gradient_descent, sigmoid
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV


In [ ]:
custom_headers = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 
    'marital-status', 'occupation', 'relationship', 'race', 'sex', 
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]
data=pd.read_csv("./Adult Data/dataset/adult.data", skipinitialspace=True, names=custom_headers, header=None)

# Preprocess

In [ ]:
#Dummies for target
data=pd.get_dummies(data, columns=['income'], drop_first=True, dtype='int')
#Converting ? into never-worked
data.loc[data['workclass']=='?', 'workclass']='Never-worked'
data.drop(columns=['fnlwgt','education'], inplace=True)

In [ ]:
X=data.drop(columns=["income_>50K"])
y=data["income_>50K"]

In [ ]:
numeric_features=['education-num',"capital-gain","capital-loss",'hours-per-week','age']
cat_fetures=['workclass','marital-status','occupation','relationship','race','sex','native-country']

In [ ]:
numerical_pipeline=Pipeline(steps=[('std_scaler',StandardScaler())])
categorical_pipeline=Pipeline(steps=[('encoder',OneHotEncoder(handle_unknown='ignore'))])
transformer=ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numeric_features),
    ('cat', categorical_pipeline, cat_fetures)  
])  

In [ ]:
main_pipeline = Pipeline(
    [("preprocessor", transformer), ("classifier", LogisticRegression())]
)

In [ ]:
cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
param_grid = [
    {
        "classifier": [
            LogisticRegression(penalty="l2", solver="lbfgs", random_state=42),
            LogisticRegression(penalty="l1", solver="liblinear", random_state=42),
        ],
        "classifier__C": [0.1, 1.0, 10.0],
    },
    {
        "classifier": [SVC()],
        "classifier__C":[0.1, 1.0, 10.0],
        "classifier__kernel": ["linear", "rbf"],
    },
    {
        "classifier": [RandomForestClassifier()],
        "classifier__n_estimators":[50,100,150],
        "classifier__max_depth": [None, 3,5,10],
    },
    {
        "classifier": [GradientBoostingClassifier(random_state=42)],
        "classifier__n_estimators":[50,100,150],
        "classifier__learning_rate": [0.01, 0.1, 0.2],
        "classifier__max_depth":[None, 3,5,10],
    }
]

In [ ]:
grid_search=GridSearchCV(estimator=main_pipeline, cv=cv, param_grid=param_grid)
grid_search.fit(X,y)